# GraphRAG with Neo4j

Ask questions about the files in `documents/` and get answers based only on them.

**Steps:** read files → split into chunks → embed chunks → store in Neo4j → extract entities and relationships (Groq) → answer questions

**Stored in Neo4j**
```
(:DocumentChunk {id, source, chunk_index, text, embedding, embedding_status, graph_status})
(:DocumentChunk)-[:MENTIONS]->(:Entity {key, name, type})
(:Entity)-[:RELATED_TO {type}]->(:Entity)
```

* **Run all cells from top to bottom.** Finished work is skipped, so re-running is safe and adds no duplicates.
* **`GROQ_ENABLED = False`** (in section 2): Groq is never called. Search still works, but there's no graph extraction and no written answers.
* **To ask your own question,** use the last cell.

## 1. Imports

In [ ]:
# Install once if needed:  %pip install python-dotenv neo4j sentence-transformers groq pypdf python-docx

import hashlib
import json
import os
import re
import time
from pathlib import Path

from docx import Document                              # reads .docx files
from dotenv import find_dotenv, load_dotenv            # reads the .env file
from neo4j import GraphDatabase                        # talks to Neo4j
from pypdf import PdfReader                            # reads .pdf files
from sentence_transformers import SentenceTransformer  # turns text into embeddings

## 2. Settings

In [ ]:
# Read NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD and GROQ_API_KEY from the .env file
load_dotenv(find_dotenv(usecwd=True), override=True)

GROQ_ENABLED = True          # False = never call Groq
GROQ_MODEL = "openai/gpt-oss-20b"

NEO4J_DATABASE = os.getenv("NEO4J_DATABASE") or None   # optional; None = the instance's default database
DOCUMENTS_DIR = Path("documents")

CHUNK_SIZE = 800       # max characters per chunk
CHUNK_OVERLAP = 150    # characters shared by neighbouring chunks
TOP_K = 5              # chunks retrieved per question
MIN_SCORE = 0.6        # ignore chunks less similar than this (0 to 1)

NOT_FOUND = "I couldn't find the answer in the uploaded documents."

## 3. Connections
Neo4j, the embedding model and Groq. Groq is only set up when `GROQ_ENABLED` is `True`.

In [ ]:
# --- Neo4j ---
driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
    notifications_min_severity="OFF",     # hide harmless server notices
)
driver.verify_connectivity()


def run_query(query, **params):
    """Run a Cypher query and return the rows as a list of dictionaries."""
    records, _, _ = driver.execute_query(query, params, database_=NEO4J_DATABASE)
    return [record.data() for record in records]


# --- Embedding model: turns text into 384 numbers; similar meaning gives similar numbers ---
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def embed(texts):
    """Return one 384-number embedding per text."""
    return model.encode(texts, normalize_embeddings=True).tolist()


# --- Groq ---
groq_client = None
if GROQ_ENABLED:
    from groq import Groq
    # max_retries=5: when Groq says "rate limit, try again in N seconds",
    # the library waits and retries automatically (up to 5 times).
    groq_client = Groq(api_key=os.environ["GROQ_API_KEY"], max_retries=5)


def ask_groq(prompt):
    """Send a prompt to Groq and return its reply text."""
    if groq_client is None:
        raise RuntimeError("Groq is disabled (GROQ_ENABLED = False).")
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,             # most consistent replies
        reasoning_effort="low",    # think briefly = fewer tokens (remove for non-reasoning models)
    )
    return response.choices[0].message.content or ""


print("Neo4j connected. Groq enabled:", GROQ_ENABLED)

## 4. Read and split documents

In [ ]:
def read_document(path):
    """Return the text of a PDF, DOCX or TXT file."""
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        return "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
    if suffix == ".docx":
        return "\n".join(paragraph.text for paragraph in Document(path).paragraphs)
    if suffix == ".txt":
        return path.read_text(encoding="utf-8", errors="replace")
    return ""


def split_text(text):
    """Split text into overlapping chunks of at most CHUNK_SIZE characters, cutting only at spaces."""
    text = re.sub(r"\s+", " ", text).strip()          # one space between words
    chunks, start = [], 0
    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        if end < len(text):                            # move the end back to a space
            space = text.rfind(" ", start + CHUNK_SIZE // 2, end)
            if space != -1:
                end = space
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = max(end - CHUNK_OVERLAP, start + 1)    # step back to create the overlap
        if text[start - 1] != " ":                     # begin at the start of a word
            space = text.find(" ", start, end)
            if space != -1:
                start = space + 1
    return [chunk for chunk in chunks if chunk]


def make_chunk_id(source, index, text):
    """Same file + position + text always gives the same ID (so chunks are never stored twice)."""
    return hashlib.sha256(f"{source}::{index}::{text}".encode("utf-8")).hexdigest()


documents = [p for p in sorted(DOCUMENTS_DIR.iterdir()) if p.suffix.lower() in {".pdf", ".docx", ".txt"}]
print("Documents:", [p.name for p in documents])

## 5. Neo4j constraints and vector index
The constraints prevent duplicate chunks and entities. The vector index makes similarity search fast. `IF NOT EXISTS` makes this safe to re-run.

In [ ]:
run_query("CREATE CONSTRAINT document_chunk_id IF NOT EXISTS FOR (c:DocumentChunk) REQUIRE c.id IS UNIQUE")
run_query("CREATE CONSTRAINT entity_key IF NOT EXISTS FOR (e:Entity) REQUIRE e.key IS UNIQUE")
run_query("""
    CREATE VECTOR INDEX chunk_vector_index IF NOT EXISTS
    FOR (c:DocumentChunk) ON c.embedding
    OPTIONS {indexConfig: {`vector.dimensions`: 384, `vector.similarity_function`: 'cosine'}}
""")


def wait_for_index():
    """Wait (up to 2 minutes) until the vector index is ready to search."""
    for _ in range(60):
        rows = run_query("SHOW VECTOR INDEXES YIELD name, state WHERE name = 'chunk_vector_index' RETURN state")
        if rows and rows[0]["state"] == "ONLINE":
            return
        time.sleep(2)
    raise TimeoutError("The vector index is not online.")

## 6. Store chunks and embeddings
Only chunks that aren't stored yet are embedded and saved. New chunks start with `graph_status = 'pending'`.

In [ ]:
def ingest_file(path):
    """Read, chunk, embed and store one file."""
    chunks = split_text(read_document(path))
    rows = [{"id": make_chunk_id(path.name, i, text), "source": path.name, "chunk_index": i, "text": text}
            for i, text in enumerate(chunks)]

    # Skip chunks that are already stored with an embedding
    stored = run_query("MATCH (c:DocumentChunk) WHERE c.id IN $ids AND c.embedding_status = 'done' RETURN c.id AS id",
                       ids=[row["id"] for row in rows])
    stored_ids = {row["id"] for row in stored}
    new_rows = [row for row in rows if row["id"] not in stored_ids]

    if new_rows:
        for row, vector in zip(new_rows, embed([row["text"] for row in new_rows])):
            row["embedding"] = vector
        # MERGE finds the chunk by id or creates it, so it is never duplicated
        run_query("""
            UNWIND $rows AS row
            MERGE (c:DocumentChunk {id: row.id})
            ON CREATE SET c.graph_status = 'pending'
            SET c.source = row.source, c.chunk_index = row.chunk_index, c.text = row.text,
                c.embedding = row.embedding, c.embedding_status = 'done'
        """, rows=new_rows)

    print(f"{path.name}: {len(new_rows)} new chunks, {len(stored_ids)} already stored")

## 7. Graph extraction with Groq
For every chunk that isn't `done`, Groq lists its entities and relationships. They are saved with `MERGE`, so the same entity is reused across chunks. A chunk that fails is retried on the next run.

In [ ]:
EXTRACTION_PROMPT = """Extract the important entities and relationships from the text below.
Return ONLY a JSON object like this:
{"entities": [{"name": "Transformer", "type": "Model"}],
 "relationships": [{"source": "Transformer", "relation": "USES", "target": "Self-attention"}]}
Rules: use only facts stated in the text, at most 10 entities and 10 relationships,
and every relationship source/target must also be listed in "entities".

Text:
"""


def tidy(name):
    """'  Self-Attention. ' -> 'Self-Attention' (removes extra spaces and edge punctuation)."""
    return " ".join(str(name).split()).strip(" .,;:'\"")


def extract_graph(text):
    """Ask Groq for the entities and relationships in a chunk. Returns (entities, relationships)."""
    reply = ask_groq(EXTRACTION_PROMPT + text)

    # Take the {...} part of the reply (ignores ```json fences or extra words) and read it as JSON
    match = re.search(r"\{.*\}", reply, flags=re.DOTALL)
    if not match:
        raise ValueError("no JSON in Groq's reply")
    data = json.loads(match.group(0))

    # Entities: the key is the lower-case name, so "OpenAI" and "openai" become one node
    entities = {}
    for item in data.get("entities", []):
        name = tidy(item.get("name", ""))
        if name:
            entities[name.lower()] = {"key": name.lower(), "name": name, "type": tidy(item.get("type", "")) or "Concept"}

    # Relationships: keep only links between two of the entities above
    relationships = []
    for item in data.get("relationships", []):
        source = tidy(item.get("source", "")).lower()
        target = tidy(item.get("target", "")).lower()
        relation = re.sub(r"[^A-Z0-9]+", "_", tidy(item.get("relation", "")).upper()).strip("_")  # "is part of" -> IS_PART_OF
        if source in entities and target in entities and relation:
            relationships.append({"source": source, "target": target, "type": relation})

    return list(entities.values()), relationships


def save_graph(chunk_id, entities, relationships):
    """Save entities and relationships and mark the chunk done, all in one query (all or nothing)."""
    run_query("""
        MATCH (c:DocumentChunk {id: $chunk_id})
        FOREACH (ent IN $entities |
            MERGE (e:Entity {key: ent.key})
            ON CREATE SET e.name = ent.name, e.type = ent.type
            MERGE (c)-[:MENTIONS]->(e))
        FOREACH (rel IN $relationships |
            MERGE (a:Entity {key: rel.source})
            MERGE (b:Entity {key: rel.target})
            MERGE (a)-[:RELATED_TO {type: rel.type}]->(b))
        SET c.graph_status = 'done'
    """, chunk_id=chunk_id, entities=entities, relationships=relationships)


def build_graph():
    """Extract the graph for every chunk that is not done yet."""
    if not GROQ_ENABLED:
        print("Groq is disabled: graph extraction skipped.")
        return

    chunks = run_query("""
        MATCH (c:DocumentChunk) WHERE c.graph_status <> 'done'
        RETURN c.id AS id, c.source AS source, c.chunk_index AS chunk_index, c.text AS text
        ORDER BY c.source, c.chunk_index
    """)
    print(len(chunks), "chunks need graph extraction")

    for chunk in chunks:
        label = f"{chunk['source']} #{chunk['chunk_index']}"
        try:
            entities, relationships = extract_graph(chunk["text"])
            save_graph(chunk["id"], entities, relationships)
            print(f"{label}: {len(entities)} entities, {len(relationships)} relationships")
        except Exception as error:
            run_query("MATCH (c:DocumentChunk {id: $id}) SET c.graph_status = 'failed'", id=chunk["id"])
            print(f"{label}: failed ({error})")
            # 429 = Groq still rate-limited after its automatic retries (e.g. daily quota used up)
            if getattr(error, "status_code", None) == 429:
                print("Groq limit reached. Run this again later to continue.")
                break

## 8. Run the ingestion
Run this again after adding files to `documents/`.

In [ ]:
for path in documents:
    ingest_file(path)

wait_for_index()
build_graph()

print(run_query("MATCH (c:DocumentChunk) RETURN c.graph_status AS graph_status, count(*) AS chunks"))

## 9. Search: vector search and graph facts

In [ ]:
def vector_search(question):
    """Return the TOP_K chunks most similar to the question, with their source file and score."""
    return run_query("""
        CALL db.index.vector.queryNodes('chunk_vector_index', $top_k, $vector)
        YIELD node, score
        RETURN node.id AS id, node.source AS source, node.chunk_index AS chunk_index, node.text AS text, score
    """, top_k=TOP_K, vector=embed([question])[0])


def graph_facts(chunk_ids):
    """Return facts like 'Transformer -[USES]-> Self-attention' about entities in the given chunks."""
    rows = run_query("""
        MATCH (c:DocumentChunk)-[:MENTIONS]->(:Entity)-[r:RELATED_TO]-(:Entity)
        WHERE c.id IN $chunk_ids
        RETURN DISTINCT startNode(r).name AS source, r.type AS relation, endNode(r).name AS target
        LIMIT 30
    """, chunk_ids=chunk_ids)
    return [f"{row['source']} -[{row['relation']}]-> {row['target']}" for row in rows]

## 10. Answer questions
`ask_question()` finds the relevant chunks and graph facts, then Groq answers **only** from them. With Groq disabled, it shows the matching passages instead of an answer.

In [ ]:
ANSWER_PROMPT = """Answer the question using ONLY the document excerpts and graph facts below.
Cite the source file name in square brackets, e.g. [report.pdf].
If they do not contain enough information, reply with exactly:
I couldn't find the answer in the uploaded documents.

Question: {question}

Document excerpts:
{context}

Graph facts:
{facts}
"""


def ask_question(question):
    """Answer a question from the uploaded documents, print it with its sources, and return the answer."""
    # 1. Find the relevant chunks (weak matches are ignored) and the graph facts about them
    chunks = [hit for hit in vector_search(question) if hit["score"] >= MIN_SCORE]
    facts = graph_facts([hit["id"] for hit in chunks])
    sources = sorted({hit["source"] for hit in chunks})

    # 2. Get the answer
    if not GROQ_ENABLED:
        answer = None                      # never invent an answer
    elif not chunks:
        answer = NOT_FOUND                 # nothing relevant: no need to call Groq
    else:
        context = "\n\n".join(f"[{hit['source']}]\n{hit['text']}" for hit in chunks)
        answer = ask_groq(ANSWER_PROMPT.format(question=question, context=context, facts="\n".join(facts) or "(none)"))
        if "couldn't find the answer" in answer.replace("’", "'").lower():
            answer = NOT_FOUND             # always use the exact sentence

    # 3. Print the result
    print("Question:", question)
    print("Answer:  ", answer or "(answer generation is disabled: GROQ_ENABLED = False)")
    print("Sources: ", ", ".join(sources) or "(none)")
    print("Graph facts used:", len(facts))
    if not GROQ_ENABLED:                   # show the matching passages instead
        for hit in chunks:
            print(f"\n[{hit['score']:.2f}] {hit['source']} #{hit['chunk_index']}: {hit['text'][:300]}...")
    return answer

## 11. Ask your own question
Change the text and run the cell (Shift+Enter). Add more cells for more questions.

In [ ]:
ask_question("Who co-authored AlexNet and later co-founded OpenAI?")